# Plan B: SD 1.5 UNet full fine-tuning (Colab)

로컬 GPU(GTX 1660 SUPER, 6GB)에서 LoRA 학습이 부족할 때를 대비한 클라우드 학습 노트북.
학습 코드는 프로젝트의 `src/train/train_full.py`를 그대로 쓰고, 이 노트북은 환경 준비와 실행만 담당한다.

## 준비 (로컬 PC)
1. 변경 사항을 커밋한다 (번들에 git commit이 기록된다).
2. 번들 생성: `python scripts/pack_for_colab.py` → `outputs/colab_bundle.zip` (코드 + `outputs/cache/short` latent/임베딩 캐시, 약 750MB)
3. Google Drive의 `MyDrive/sd-engine/` 폴더에 `colab_bundle.zip`을 올린다.

## Colab
- 런타임 → 런타임 유형 변경 → GPU (A100 / L4 / T4) 선택 후 위에서부터 차례로 실행한다.
- 학습 출력(샘플, 체크포인트, 로그)은 Drive의 `MyDrive/sd-engine/outputs/`에 저장되어 세션이 끊겨도 남는다.
- 세션이 끊기면 1~5번 셀을 다시 실행한 뒤 본 학습 셀을 실행하면 마지막 체크포인트에서 이어서 학습한다 (옵티마이저 상태는 새로 시작).

## 메모리 / 시간 (추정, 실측 전)
| GPU | 설정 | 비고 |
|---|---|---|
| A100 40GB | `train_full.yaml` (배치 8, bf16) | |
| L4 24GB | + `train_full_24gb.yaml` (배치 4×2, bf16, 8-bit Adam) | |
| T4 16GB | + `train_full_t4.yaml` (배치 2×4, fp16, 8-bit Adam) | 로컬 GTX 1660처럼 fp16에서 NaN이 나면 중단됨 |

In [ ]:
# 설정
DRIVE_DIR = "/content/drive/MyDrive/sd-engine"   # colab_bundle.zip 을 올린 Drive 폴더
BUNDLE_NAME = "colab_bundle.zip"
WORK_DIR = "/content/sde"                         # 번들을 풀 로컬 경로 (캐시 읽기가 Drive보다 빠름)
SEED = 42

## 1. GPU 확인과 설정 프리셋 선택

In [ ]:
!nvidia-smi
import torch

assert torch.cuda.is_available(), "GPU 런타임이 아니다. 런타임 유형을 GPU로 바꾼다."
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1024**3
bf16 = torch.cuda.is_bf16_supported()
if not bf16:
    PRESET = ["configs/train_full_t4.yaml"]
elif vram_gb < 40:
    PRESET = ["configs/train_full_24gb.yaml"]
else:
    PRESET = []
print(f"{props.name}, VRAM {vram_gb:.1f}GB, bf16={bf16}, torch {torch.__version__} -> 추가 설정: {PRESET or '없음 (A100 기본)'}")

## 2. Drive 마운트와 번들 풀기

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, os, zipfile
os.makedirs(WORK_DIR, exist_ok=True)
with zipfile.ZipFile(f"{DRIVE_DIR}/{BUNDLE_NAME}") as zf:
    zf.extractall(WORK_DIR)
os.chdir(WORK_DIR)
print(json.load(open("BUNDLE_INFO.json")))
!ls configs outputs/cache/short

## 3. 패키지 설치
Colab에 미리 설치된 torch를 그대로 쓰고 나머지만 설치한다 (`requirements-torch.txt`는 로컬 CUDA용이라 설치하지 않음).
Colab 기본 패키지와 버전 충돌 경고가 나올 수 있다. 설치 후 런타임 재시작을 요구하면 재시작하고 2번 셀부터 다시 실행한다.

In [ ]:
!pip install -q -r requirements.txt

## 4. 환경 점검 (브라우저 검사 생략)

In [ ]:
!python scripts/check_env.py --skip-browser

## 5. Colab용 출력 경로 설정 (Drive에 저장)

In [ ]:
OUTPUT_DIR = f"{DRIVE_DIR}/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open("configs/colab_paths.yaml", "w", encoding="utf-8") as f:
    f.write(f"paths:\n  output_dir: {OUTPUT_DIR}\n")
BASE_CONFIGS = ["configs/base.yaml", "configs/train_lora.yaml", "configs/train_full.yaml", *PRESET, "configs/colab_paths.yaml"]
print(BASE_CONFIGS)

## 6. overfit 테스트 (권장, 학습 32장)
CLAUDE.md 규칙: 본 학습 전에 소규모 overfit으로 파이프라인을 확인한다. 학습 샘플(그리드 아래쪽 4행)이 정답과 거의 같아지면 통과.

In [ ]:
OVERFIT_CONFIGS = BASE_CONFIGS[:-1] + ["configs/train_full_overfit.yaml", "configs/colab_paths.yaml"]
!python -m src.train.train_full --seed {SEED} --config {" ".join(OVERFIT_CONFIGS)}

### overfit 결과 보기 (왼쪽 정답 | 오른쪽 생성, 위 2행 검증 / 아래 4행 학습)

In [ ]:
import glob
from IPython.display import Image, display

def show_run(run_name: str) -> None:
    run_dir = f"{OUTPUT_DIR}/{run_name}"
    for line in open(f"{run_dir}/samples/metrics.jsonl"):
        r = json.loads(line)
        print(f"step {r['step']:>5}: val MSE {r['val_mse']:.4f} / train MSE {r['train_mse']:.4f}")
    latest = sorted(glob.glob(f"{run_dir}/samples/step_*.png"))[-1]
    print(latest)
    display(Image(latest, width=512))

show_run("full_overfit")

## 7. 본 학습 (짧은 데이터셋 전체)
마지막 체크포인트가 Drive에 있으면 자동으로 이어서 학습한다.

In [ ]:
RUN_NAME = "full_short"
ckpts = sorted(glob.glob(f"{OUTPUT_DIR}/{RUN_NAME}/checkpoints/step_*"))
RESUME = f"--resume {ckpts[-1]}" if ckpts else ""
print("재개:", RESUME or "처음부터")
!python -m src.train.train_full --seed {SEED} --config {" ".join(BASE_CONFIGS)} {RESUME}

## 8. 학습 경과 보기

In [ ]:
show_run("full_short")
!tail -n 3 {OUTPUT_DIR}/full_short/log.jsonl

## 9. 추론
학습에 쓰지 않은 HTML을 넣어 본다. 짧은 데이터셋과 같은 형식(CLIP 77토큰 이내)이어야 한다.

In [ ]:
CKPT = sorted(glob.glob(f"{OUTPUT_DIR}/full_short/checkpoints/step_*"))[-1]
HTML = '<body style="margin:0;padding:16px;background:#def;color:#000"><h1>Hello world</h1><div style="width:50%;height:120px;background:#e33"></div></body>'
!python -m src.infer.generate --full {CKPT} --html-text '{HTML}' --out {OUTPUT_DIR}/infer/sample.png --seed {SEED}
display(Image(f"{OUTPUT_DIR}/infer/sample.png", width=512))